# Universal Video to Video Converter
Convert video files with maximum flexibility:
- **Single Video File**
- **Multiple Selected Video Files** (hold Ctrl/Shift to choose multiple)
- **Full Folder / Directory** (batch conversion)

### Supported Video Formats (19 Containers):
`.mp4`, `.mkv`, `.mov`, `.avi`, `.wmv`, `.webm`, `.flv`, `.mpeg`, `.mpg`, `.m4v`, `.3gp`, `.ts`, `.mts`, `.m2ts`, `.vob`, `.ogv`, `.asf`, `.divx`, `.f4v`

---
### Key Features:
- **Interactive Multi-Mode File/Folder Picker**: Select 1 file, multiple files, or an entire directory.
- **Input Filter & Output Selector**: Choose from all 19 video formats or filter specific source containers.
- **Intelligent Codecs**: Automatic pairing of optimal video and audio codecs (`libx264`, `libvpx-vp9`, `mpeg4`, `wmv2`, etc.).
- **Stream Copy Mode**: Lossless remuxing in seconds without re-encoding when container compatible.
- **Progress & Metrics**: Real-time progress percentage, file size analysis, and JSON audit logging.

## Step 1: Environment & FFmpeg Verification
Checks for `ffmpeg` in PATH, `.venv`, `imageio_ffmpeg`, or standard paths.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

def detect_ffmpeg():
    # 1. System PATH
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
        
    # 2. Local venv
    venv_exe = Path(r"d:\New folder\PY\.venv\Scripts\ffmpeg.exe")
    if venv_exe.is_file():
        os.environ["PATH"] = str(venv_exe.parent) + os.pathsep + os.environ.get("PATH", "")
        return str(venv_exe)

    # 3. imageio_ffmpeg bundled binary
    try:
        import imageio_ffmpeg
        loc = imageio_ffmpeg.get_ffmpeg_exe()
        if loc and os.path.isfile(loc):
            os.environ["PATH"] = str(Path(loc).parent) + os.pathsep + os.environ.get("PATH", "")
            return loc
    except Exception:
        pass

    # 4. Common locations
    common_locations = [
        r"C:\ffmpeg\bin\ffmpeg.exe",
        r"C:\ffmpeg\ffmpeg.exe",
        r"C:\Program Files\ffmpeg\bin\ffmpeg.exe",
        r"C:\ProgramData\chocolatey\bin\ffmpeg.exe",
        os.path.expanduser(r"~\AppData\Local\Microsoft\WinGet\Links\ffmpeg.exe"),
        os.path.expanduser(r"~\AppData\Local\ffmpeg\bin\ffmpeg.exe"),
    ]
    for loc in common_locations:
        if os.path.isfile(loc):
            os.environ["PATH"] = str(Path(loc).parent) + os.pathsep + os.environ.get("PATH", "")
            return loc

    return None

FFMPEG_PATH = detect_ffmpeg()
if FFMPEG_PATH:
    print(f"FFmpeg detected successfully: {FFMPEG_PATH}")
else:
    print("WARNING: FFmpeg was not detected in PATH.")
    print("Run: pip install imageio-ffmpeg")


## Step 2: Define Supported Video Formats and Codec Matrix
Supported container mappings and default codecs.

In [ ]:
VIDEO_FORMATS = [
    ".mp4", ".mkv", ".mov", ".avi", ".wmv", ".webm", ".flv", ".mpeg",
    ".mpg", ".m4v", ".3gp", ".ts", ".mts", ".m2ts", ".vob", ".ogv",
    ".asf", ".divx", ".f4v"
]

VIDEO_CODECS_MAP = {
    ".mp4": ("libx264", "aac"),
    ".mkv": ("libx264", "aac"),
    ".webm": ("libvpx-vp9", "libopus"),
    ".mov": ("libx264", "aac"),
    ".avi": ("mpeg4", "mp3"),
    ".wmv": ("wmv2", "wmav2"),
    ".flv": ("flv1", "mp3"),
    ".f4v": ("libx264", "aac"),
    ".mpeg": ("mpeg2video", "mp2"),
    ".mpg": ("mpeg2video", "mp2"),
    ".vob": ("mpeg2video", "ac3"),
    ".m4v": ("libx264", "aac"),
    ".3gp": ("libx264", "aac"),
    ".ts": ("libx264", "aac"),
    ".mts": ("libx264", "aac"),
    ".m2ts": ("libx264", "aac"),
    ".ogv": ("libtheora", "libvorbis"),
    ".asf": ("wmv2", "wmav2"),
    ".divx": ("mpeg4", "mp3")
}

print(f"Configured {len(VIDEO_FORMATS)} video formats.")


## Step 3: Interactive Selection (Single File, Multiple Files, or Full Folder)
Run this cell to select your input:
- Option [1]: Select **multiple files** (hold Ctrl or Shift in the browser window).
- Option [2]: Select a **single file**.
- Option [3]: Select a **full folder**.
- Option [4]: Type or paste path(s) manually.

In [ ]:
import tkinter as tk
from tkinter import filedialog

def pick_media_inputs(supported_formats, media_type="video"):
    selected_files = []
    source_dir = None
    is_folder = False

    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        print("\nChoose input selection mode:")
        print("  [1] Select MULTIPLE files (Hold Ctrl/Shift to choose multiple files)")
        print("  [2] Select a SINGLE file")
        print("  [3] Select a FULL FOLDER (Process all matching files in directory)")
        print("  [4] Type or paste path(s) manually")
        choice = input("Enter choice [1/2/3/4] (default: 1): ").strip()
        
        ext_filter = [(f"{media_type.title()} Files", "*" + ";*".join(supported_formats)), ("All Files", "*.*")]
        
        if choice == "2":
            single_path = filedialog.askopenfilename(title=f"Select Single {media_type.title()} File", filetypes=ext_filter)
            if single_path:
                p = Path(single_path)
                selected_files = [p]
                source_dir = p.parent
        elif choice == "3":
            folder_path = filedialog.askdirectory(title=f"Select Folder Containing {media_type.title()} Files")
            if folder_path:
                source_dir = Path(folder_path)
                is_folder = True
        elif choice == "4":
            pass
        else:
            multi_paths = filedialog.askopenfilenames(title=f"Select One or Multiple {media_type.title()} Files", filetypes=ext_filter)
            if multi_paths:
                selected_files = [Path(p) for p in multi_paths]
                source_dir = selected_files[0].parent
        root.destroy()
    except Exception:
        pass
        
    if not selected_files and not is_folder:
        raw = input(f"Enter {media_type} path(s) or folder (separate multiple files with comma): ").strip().strip('"').strip("'")
        if raw:
            if "," in raw:
                parts = [p.strip().strip('"').strip("'") for p in raw.split(",") if p.strip()]
            elif ";" in raw:
                parts = [p.strip().strip('"').strip("'") for p in raw.split(";") if p.strip()]
            else:
                parts = [raw]

            if len(parts) == 1 and Path(parts[0]).is_dir():
                is_folder = True
                source_dir = Path(parts[0])
            else:
                selected_files = [Path(p) for p in parts]
                source_dir = selected_files[0].parent

    return selected_files, is_folder, source_dir

SELECTED_FILES, IS_FOLDER, SOURCE_DIR = pick_media_inputs(VIDEO_FORMATS, media_type="video")
if not IS_FOLDER and not SELECTED_FILES:
    raise ValueError("No valid input files or folder selected.")

if IS_FOLDER:
    print(f"\nSelection Mode: Full Folder -> {SOURCE_DIR}")
elif len(SELECTED_FILES) == 1:
    print(f"\nSelection Mode: Single File -> {SELECTED_FILES[0].name}")
else:
    print(f"\nSelection Mode: Multiple Files -> {len(SELECTED_FILES)} files selected in {SOURCE_DIR}")


## Step 4: Select Input Format Filter & Target Output Format
Choose which input files to process and your target output container.

In [ ]:
print("\n" + "=" * 55)
print("SELECT INPUT FORMAT FILTER")
print("=" * 55)
print("  [0] ALL Supported Video Formats (*.*)")
for idx, ext in enumerate(VIDEO_FORMATS, 1):
    print(f"  [{idx:2d}] {ext}")
print("=" * 55)

in_choice = input("Enter input format choice [0 for all, or 1-19, or type extension] (default: 0): ").strip()
if not in_choice or in_choice == "0":
    SELECTED_INPUT_FILTER = "all"
elif in_choice.isdigit() and 1 <= int(in_choice) <= len(VIDEO_FORMATS):
    SELECTED_INPUT_FILTER = VIDEO_FORMATS[int(in_choice) - 1]
elif in_choice.startswith(".") and in_choice.lower() in VIDEO_FORMATS:
    SELECTED_INPUT_FILTER = in_choice.lower()
elif ("." + in_choice.lower()) in VIDEO_FORMATS:
    SELECTED_INPUT_FILTER = "." + in_choice.lower()
else:
    print(f"Unrecognized choice '{in_choice}'. Defaulting to 'all'.")
    SELECTED_INPUT_FILTER = "all"

print(f"\nInput filter selected: {SELECTED_INPUT_FILTER.upper()}")

print("\n" + "=" * 55)
print("SELECT TARGET OUTPUT FORMAT")
print("=" * 55)
for idx, ext in enumerate(VIDEO_FORMATS, 1):
    print(f"  [{idx:2d}] {ext}")
print("=" * 55)

out_choice = input("Enter target output format [1-19, or type extension e.g. .mp4] (default: 1 [.mp4]): ").strip()
if not out_choice or out_choice == "1":
    TARGET_OUTPUT_FORMAT = ".mp4"
elif out_choice.isdigit() and 1 <= int(out_choice) <= len(VIDEO_FORMATS):
    TARGET_OUTPUT_FORMAT = VIDEO_FORMATS[int(out_choice) - 1]
elif out_choice.startswith(".") and out_choice.lower() in VIDEO_FORMATS:
    TARGET_OUTPUT_FORMAT = out_choice.lower()
elif ("." + out_choice.lower()) in VIDEO_FORMATS:
    TARGET_OUTPUT_FORMAT = "." + out_choice.lower()
else:
    print(f"Unrecognized choice '{out_choice}'. Defaulting to '.mp4'.")
    TARGET_OUTPUT_FORMAT = ".mp4"

print(f"Target output format selected: {TARGET_OUTPUT_FORMAT}")


## Step 5: Encoding Preset & Destination Setup
Configure quality presets and scan target files.

In [ ]:
print("\nSelect Video Quality / Mode:")
print("  [1] High Quality / Balanced (Recommended: CRF 23, preset medium)")
print("  [2] Best Quality / Near Lossless (CRF 18, preset slow)")
print("  [3] Fast / Compact File Size (CRF 28, preset fast)")
print("  [4] Stream Copy (Lossless & Instant - skips re-encoding if compatible)")
mode_choice = input("Enter choice [1/2/3/4] (default: 1): ").strip()

STREAM_COPY = False
CRF_VALUE = "23"
PRESET_VALUE = "medium"

if mode_choice == "2":
    CRF_VALUE = "18"
    PRESET_VALUE = "slow"
elif mode_choice == "3":
    CRF_VALUE = "28"
    PRESET_VALUE = "fast"
elif mode_choice == "4":
    STREAM_COPY = True

# Gather files to convert
if IS_FOLDER:
    if SELECTED_INPUT_FILTER == "all":
        target_exts = set(VIDEO_FORMATS)
    else:
        target_exts = {SELECTED_INPUT_FILTER}
    
    files_to_convert = [
        f for f in SOURCE_DIR.iterdir()
        if f.is_file() and f.suffix.lower() in target_exts and f.suffix.lower() != TARGET_OUTPUT_FORMAT
    ]
    if not files_to_convert:
        files_to_convert = [
            f for f in SOURCE_DIR.rglob("*")
            if f.is_file() and f.suffix.lower() in target_exts and f.suffix.lower() != TARGET_OUTPUT_FORMAT
        ]
else:
    if SELECTED_INPUT_FILTER != "all":
        files_to_convert = [
            f for f in SELECTED_FILES
            if f.is_file() and f.suffix.lower() == SELECTED_INPUT_FILTER and f.suffix.lower() != TARGET_OUTPUT_FORMAT
        ]
    else:
        files_to_convert = [
            f for f in SELECTED_FILES
            if f.is_file() and f.suffix.lower() in VIDEO_FORMATS and f.suffix.lower() != TARGET_OUTPUT_FORMAT
        ]

if not files_to_convert:
    raise ValueError(f"No video files matching filter '{SELECTED_INPUT_FILTER}' found for conversion.")

clean_out_name = f"converted_videos_{TARGET_OUTPUT_FORMAT.lstrip('.')}"
OUTPUT_DIR = SOURCE_DIR / clean_out_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nReady to convert {len(files_to_convert)} video file(s).")
print(f"Destination folder: {OUTPUT_DIR}")


## Step 6: Conversion Engine with Real-Time Progress
Executes conversion using FFmpeg with live percentage progress.

In [ ]:
import time
import re
import json

def get_duration(file_path, ffmpeg_bin):
    if not ffmpeg_bin:
        return None
    try:
        proc = subprocess.run(
            [ffmpeg_bin, "-hide_banner", "-i", str(file_path)],
            stderr=subprocess.PIPE,
            stdout=subprocess.DEVNULL,
            text=True,
            encoding="utf-8",
            errors="replace"
        )
        m = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", proc.stderr)
        if m:
            h, mins, s = float(m.group(1)), float(m.group(2)), float(m.group(3))
            return h * 3600 + mins * 60 + s
    except Exception:
        pass
    return None

def convert_single_video(src_file, dst_file, target_ext, ffmpeg_bin="ffmpeg", stream_copy=False, crf="23", preset="medium"):
    if stream_copy:
        cmd = [ffmpeg_bin, "-y", "-i", str(src_file), "-c", "copy", "-nostats", "-progress", "pipe:1", str(dst_file)]
    else:
        v_codec, a_codec = VIDEO_CODECS_MAP.get(target_ext.lower(), ("libx264", "aac"))
        cmd = [ffmpeg_bin, "-y", "-i", str(src_file), "-c:v", v_codec]
        
        if v_codec in ("libx264", "libx265"):
            cmd.extend(["-crf", str(crf), "-preset", preset])
            if target_ext.lower() == ".mp4":
                cmd.extend(["-pix_fmt", "yuv420p"])
        elif v_codec == "libvpx-vp9":
            cmd.extend(["-crf", str(crf), "-b:v", "0"])
        elif v_codec in ("mpeg4", "mpeg2video"):
            cmd.extend(["-q:v", "3"])
            
        cmd.extend(["-c:a", a_codec])
        if a_codec in ("aac", "libmp3lame", "mp3", "libvorbis"):
            cmd.extend(["-b:a", "192k"])
        elif a_codec == "libopus":
            cmd.extend(["-b:a", "128k"])
            
        cmd.extend(["-nostats", "-progress", "pipe:1", str(dst_file)])
        
    duration_sec = get_duration(src_file, ffmpeg_bin)
    start_time = time.time()
    last_reported = -1
    log_tail = []

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        universal_newlines=True,
        bufsize=1
    )

    for line in proc.stdout:
        line_str = line.strip()
        if not line_str:
            continue
        log_tail.append(line_str)
        if len(log_tail) > 100:
            log_tail.pop(0)

        if duration_sec and line_str.startswith("out_time_ms="):
            try:
                ms_val = float(line_str.split("=", 1)[1])
                curr_sec = ms_val / 1_000_000.0
                pct = min(100.0, (curr_sec / duration_sec) * 100.0)
                whole = int(pct)
                if whole >= last_reported + 10:
                    last_reported = whole
                    print(f"    Progress: {pct:5.1f}% [{curr_sec:.1f}s / {duration_sec:.1f}s]")
            except Exception:
                pass
        elif line_str == "progress=end":
            print("    Progress: 100.0% [Completed]")

    ret = proc.wait()
    if ret != 0:
        tail_msg = "\n".join(log_tail[-20:])
        raise RuntimeError(f"FFmpeg conversion failed (code {ret}):\n{tail_msg}")

    elapsed = time.time() - start_time
    return {
        "input": str(src_file),
        "output": str(dst_file),
        "elapsed_seconds": round(elapsed, 2),
        "input_size_bytes": src_file.stat().st_size,
        "output_size_bytes": dst_file.stat().st_size if dst_file.exists() else 0,
        "status": "Success"
    }

conversion_records = []
total_count = len(files_to_convert)

print(f"\nStarting conversion of {total_count} video(s)...\n" + "=" * 55)
for i, fpath in enumerate(files_to_convert, 1):
    out_filename = f"{fpath.stem}{TARGET_OUTPUT_FORMAT}"
    dest_path = OUTPUT_DIR / out_filename
    print(f"[{i}/{total_count}] Converting: {fpath.name} -> {out_filename}")
    try:
        rec = convert_single_video(
            src_file=fpath,
            dst_file=dest_path,
            target_ext=TARGET_OUTPUT_FORMAT,
            ffmpeg_bin=FFMPEG_PATH,
            stream_copy=STREAM_COPY,
            crf=CRF_VALUE,
            preset=PRESET_VALUE
        )
        conversion_records.append(rec)
        print(f"  OK: Finished in {rec['elapsed_seconds']}s")
    except Exception as exc:
        print(f"  ERROR on {fpath.name}: {exc}")
        conversion_records.append({
            "input": str(fpath),
            "output": str(dest_path),
            "status": "Failed",
            "error": str(exc)
        })

print("=" * 55 + "\nAll conversion tasks completed!")


## Step 7: Summary & Audit Report
Inspect before/after size comparisons and export JSON report.

In [ ]:
total_in_bytes = 0
total_out_bytes = 0
total_time = 0

print("\n" + "=" * 70)
print(f"{'Source File':<28} | {'In Size':<10} | {'Out Size':<10} | {'Ratio':<6} | {'Time'}")
print("-" * 70)

for rec in conversion_records:
    if rec.get("status") == "Success":
        in_mb = rec["input_size_bytes"] / (1024 * 1024)
        out_mb = rec["output_size_bytes"] / (1024 * 1024)
        ratio = (out_mb / in_mb) if in_mb > 0 else 1.0
        fname = Path(rec["input"]).name
        if len(fname) > 27:
            fname = fname[:24] + "..."
        print(f"{fname:<28} | {in_mb:7.2f} MB | {out_mb:7.2f} MB | {ratio:5.2f}x | {rec['elapsed_seconds']}s")
        total_in_bytes += rec["input_size_bytes"]
        total_out_bytes += rec["output_size_bytes"]
        total_time += rec["elapsed_seconds"]
    else:
        fname = Path(rec["input"]).name
        print(f"{fname:<28} | FAILED: {rec.get('error', 'Unknown')[:35]}")

print("=" * 70)
tot_in_mb = total_in_bytes / (1024 * 1024)
tot_out_mb = total_out_bytes / (1024 * 1024)
print(f"Total Input Size:  {tot_in_mb:.2f} MB")
print(f"Total Output Size: {tot_out_mb:.2f} MB")
if tot_in_mb > 0:
    print(f"Overall Ratio:     {(tot_out_mb / tot_in_mb):.2f}x")
print(f"Total Duration:    {total_time:.2f} seconds")
print(f"Output Directory:  {OUTPUT_DIR}")

# Save JSON report
report_path = OUTPUT_DIR / "conversion_summary.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump({
        "conversion_type": "video_to_video",
        "target_format": TARGET_OUTPUT_FORMAT,
        "input_filter": SELECTED_INPUT_FILTER,
        "records": conversion_records
    }, f, indent=2)
print(f"Saved audit report to: {report_path}")
